# Early Solana Rug-Pull Prediction Pipeline

Clean, leakage-safe pipeline built from `solrpds_v5_v0.ipynb`.

Goals:
- labels may use full lifecycle data
- features must come from an early observation window only
- no random splits
- validation is calendar-based around the July 1, 2024 event date
- 2025 is reserved for true forward prediction/testing when labels are available

Note: the current raw files mix schemas. The 2021-2024 files are pool-level lifecycle snapshots, while the 2025 files are monthly mint/project summaries. This notebook normalizes both, but it refuses to use full-lifecycle label columns as features.


In [25]:
import json
import math
import pickle
import warnings
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

RAW_DIR = Path("data/raw")
OUT_DIR = Path("data/early_prediction")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EVENT_DATE = pd.Timestamp("2024-07-01")
EARLY_WINDOW_MINUTES = 30
EARLY_WINDOW = pd.Timedelta(minutes=EARLY_WINDOW_MINUTES)
RUG_RATIO_MIN = 1.5
RUG_LIFESPAN_MAX_MIN = 360
RANDOM_STATE = 42
PREDICTION_THRESHOLD = 0.02

LABEL_ONLY_COLUMNS = {
    "INACTIVITY_STATUS",
    "inactivity_status",
    "ADD_TO_REMOVE_RATIO",
    "add_to_remove_ratio",
    "lifespan_min",
    "lifespan_days",
    "LAST_POOL_ACTIVITY_TIMESTAMP",
    "last_activity_time",
    "LAST_SWAP_TIMESTAMP",
    "LAST_SWAP_TX_ID",
    "TOTAL_ADDED_LIQUIDITY",
    "TOTAL_REMOVED_LIQUIDITY",
    "NUM_LIQUIDITY_ADDS",
    "NUM_LIQUIDITY_REMOVES",
}

print(f"Early window: first {EARLY_WINDOW_MINUTES} minutes")
print(f"Outputs: {OUT_DIR}")


Early window: first 30 minutes
Outputs: data\early_prediction


## Step 1 - Load And Normalize

The loader maps known Dune/Solana schemas into canonical names:

- `pool_id`
- `token_mint`
- `creator`
- `timestamp`
- lifecycle label columns retained only for labeling
- event-like metrics retained only when they are genuinely available


In [26]:
def parse_ts(series):
    return pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert(None)


def first_existing(df, names):
    for name in names:
        if name in df.columns:
            return name
    return None


def normalize_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    out = pd.DataFrame(index=df.index)
    out["_source"] = path.name
    out["_raw_schema"] = ",".join(df.columns)

    pool_col = first_existing(df, ["pool_id", "LIQUIDITY_POOL_ADDRESS", "liquidity_pool_address"])
    mint_col = first_existing(df, ["token_mint", "MINT", "mint"])
    creator_col = first_existing(df, ["creator", "CREATOR_WALLET", "creator_wallet", "project"])
    ts_col = first_existing(df, ["timestamp", "FIRST_POOL_ACTIVITY_TIMESTAMP", "first_activity_time"])
    last_col = first_existing(df, ["LAST_POOL_ACTIVITY_TIMESTAMP", "last_activity_time"])

    out["pool_id"] = df[pool_col].astype(str) if pool_col else pd.Series([np.nan] * len(df), dtype="object")
    out["token_mint"] = df[mint_col].astype(str) if mint_col else pd.Series([np.nan] * len(df), dtype="object")
    out["creator"] = df[creator_col].astype(str) if creator_col else pd.Series([np.nan] * len(df), dtype="object")
    out["timestamp"] = parse_ts(df[ts_col]) if ts_col else pd.NaT
    out["last_timestamp"] = parse_ts(df[last_col]) if last_col else pd.NaT

    # If no pool id exists, keep rows identifiable without pretending mint-level rows are pools.
    pool_as_text = out["pool_id"].astype("string")
    missing_pool = out["pool_id"].isna() | pool_as_text.str.lower().isin(["nan", "none", ""])
    if missing_pool.any():
        out.loc[missing_pool, "pool_id"] = (
            out.loc[missing_pool, "token_mint"].fillna("unknown_mint")
            + "::"
            + out.loc[missing_pool, "creator"].fillna("unknown_creator")
            + "::"
            + out.loc[missing_pool, "_source"]
        )

    numeric_map = {
        "total_added_liquidity": ["TOTAL_ADDED_LIQUIDITY", "total_added_liquidity"],
        "total_removed_liquidity": ["TOTAL_REMOVED_LIQUIDITY", "total_removed_liquidity"],
        "num_liquidity_adds": ["NUM_LIQUIDITY_ADDS", "num_liquidity_adds"],
        "num_liquidity_removes": ["NUM_LIQUIDITY_REMOVES", "num_liquidity_removes"],
        "add_to_remove_ratio": ["ADD_TO_REMOVE_RATIO", "add_to_remove_ratio"],
        "num_buy_swaps": ["num_buy_swaps"],
        "num_sell_swaps": ["num_sell_swaps"],
        "num_total_swaps": ["num_total_swaps"],
        "buy_volume_usd": ["buy_volume_usd"],
        "sell_volume_usd": ["sell_volume_usd"],
        "total_volume_usd": ["total_volume_usd"],
        "lifespan_days": ["lifespan_days"],
    }

    for out_col, candidates in numeric_map.items():
        source = first_existing(df, candidates)
        out[out_col] = pd.to_numeric(df[source], errors="coerce") if source else np.nan

    status_col = first_existing(df, ["INACTIVITY_STATUS", "inactivity_status"])
    out["inactivity_status"] = df[status_col] if status_col else np.nan

    if out["last_timestamp"].notna().any():
        out["lifespan_min"] = (out["last_timestamp"] - out["timestamp"]).dt.total_seconds() / 60
    elif out["lifespan_days"].notna().any():
        out["lifespan_min"] = out["lifespan_days"] * 24 * 60
    else:
        out["lifespan_min"] = np.nan

    # Event-level detection: true event rows usually have one timestamped event/tx per row.
    event_cols = {"tx_id", "signature", "user", "wallet", "amount", "volume_usd", "event_type"}
    out["_looks_event_level"] = len(event_cols & set(df.columns)) > 0
    for optional in ["tx_id", "signature", "user", "wallet", "amount", "volume_usd", "event_type"]:
        out[optional] = df[optional] if optional in df.columns else np.nan

    return out


raw_files = sorted(RAW_DIR.glob("*.csv"))
if not raw_files:
    raise FileNotFoundError(f"No CSV files found in {RAW_DIR}")

raw = pd.concat([normalize_file(path) for path in raw_files], ignore_index=True)
raw = raw.dropna(subset=["pool_id", "token_mint", "timestamp"]).reset_index(drop=True)

print(f"Files loaded: {len(raw_files)}")
print(f"Rows loaded: {len(raw):,}")
print(f"Date range: {raw['timestamp'].min()} -> {raw['timestamp'].max()}")
print("Event-level rows:", int(raw["_looks_event_level"].sum()))
display(raw.head())


Files loaded: 16
Rows loaded: 128,304
Date range: 2021-02-14 21:09:21 -> 2026-04-29 16:29:07
Event-level rows: 0


,_source,_raw_schema,pool_id,token_mint,creator,timestamp,last_timestamp,total_added_liquidity,total_removed_liquidity,num_liquidity_adds,...,inactivity_status,lifespan_min,_looks_event_level,tx_id,signature,user,wallet,amount,volume_usd,event_type
0,2021.csv,"LIQUIDITY_POOL_ADDRESS,MINT,TOTAL_ADDED_LIQUID...",821L241gacEqKJiCp9H7E9rqRXB31WoHLBqwhn8NMyWS,EPjFWdd5AufqSSqeM2qN1xzybapC8G4wEGGkZwyTDt1v,NaN,2021-12-30 21:55:04,2021-12-30 22:26:30,1.314267e+02,3.787952e+01,4.0,...,Active,31.433333,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021.csv,"LIQUIDITY_POOL_ADDRESS,MINT,TOTAL_ADDED_LIQUID...",821L241gacEqKJiCp9H7E9rqRXB31WoHLBqwhn8NMyWS,5L87fjh5XZWERN4UGbK62TM1funxFvXSRUGmvbHBGqn1,NaN,2021-12-30 21:55:04,2021-12-30 22:26:30,6.467274e+08,7.470788e+08,4.0,...,Active,31.433333,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021.csv,"LIQUIDITY_POOL_ADDRESS,MINT,TOTAL_ADDED_LIQUID...",3nesbuhAwCMGtsyypYtg4oRPwJ3FmHnytC5bqskhPh1x,A94X2fRy3wydNShU4dRaDyap2UuoeWJGWyATtyp61WZf,NaN,2021-12-30 15:52:23,2021-12-30 19:59:48,5.015108e+06,6.499965e+00,6.0,...,Active,247.416667,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021.csv,"LIQUIDITY_POOL_ADDRESS,MINT,TOTAL_ADDED_LIQUID...",3nesbuhAwCMGtsyypYtg4oRPwJ3FmHnytC5bqskhPh1x,9QBTKuSCDaJjtxYnYcVzoiKENMdJ5DRei5ZUCEeWyZnj,NaN,2021-12-30 15:52:23,2021-12-30 20:30:32,5.000006e+06,1.282729e+04,2.0,...,Active,278.150000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021.csv,"LIQUIDITY_POOL_ADDRESS,MINT,TOTAL_ADDED_LIQUID...",85anbeqV36qqMvdVQw2ca7YRdJoBQ95DoUUFW9vC4Bav,meebAU3nZrU5PbUt3dVK6ExgbNWCUAkV7C3DaJKMZZ4,NaN,2021-12-30 00:57:19,2021-12-30 06:35:34,2.643479e+00,2.955737e+01,2.0,...,Active,338.250000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 2 - Label On Full Data Only

Labels intentionally use full lifecycle fields. These fields are removed before modeling.


In [27]:
def normalize_inactive(value):
    if pd.isna(value):
        return False
    text = str(value).strip().lower()
    return text in {"inactive", "1", "true", "yes"}


label_frame = raw.sort_values("last_timestamp").drop_duplicates("pool_id", keep="last").copy()
label_frame["is_rug"] = (
    label_frame["inactivity_status"].map(normalize_inactive)
    & (label_frame["add_to_remove_ratio"] > RUG_RATIO_MIN)
    & (label_frame["lifespan_min"] < RUG_LIFESPAN_MAX_MIN)
)

label_frame["has_label"] = (
    label_frame["inactivity_status"].notna()
    & label_frame["add_to_remove_ratio"].notna()
    & label_frame["lifespan_min"].notna()
)
label_frame["is_rug"] = label_frame["is_rug"].astype("Int64")

labels = label_frame[["pool_id", "is_rug", "has_label"]].copy()

print(f"Labeled pools: {labels['has_label'].sum():,} / {len(labels):,}")
print(labels.loc[labels["has_label"], "is_rug"].value_counts(dropna=False))


Labeled pools: 63,520 / 75,520
is_rug
0    62431
1     1089
Name: count, dtype: Int64


## Step 3 - Early Window Features

Feature engineering uses only rows where `timestamp <= first_pool_timestamp + X minutes`.

If the raw files are not event-level, the notebook still creates leakage-safe metadata features, but it leaves transaction/volume early metrics as missing instead of using full lifecycle totals.


In [28]:
pool_start = raw.groupby("pool_id", as_index=False)["timestamp"].min().rename(columns={"timestamp": "pool_start"})
events = raw.merge(pool_start, on="pool_id", how="left")
events["minutes_since_start"] = (events["timestamp"] - events["pool_start"]).dt.total_seconds() / 60
early_events = events[
    (events["minutes_since_start"] >= 0)
    & (events["minutes_since_start"] <= EARLY_WINDOW_MINUTES)
].copy()

has_event_level = bool(early_events["_looks_event_level"].any())

base = (
    events.sort_values("timestamp")
    .groupby("pool_id", as_index=False)
    .first()[["pool_id", "token_mint", "creator", "pool_start"]]
)
base["pool_start_year"] = base["pool_start"].dt.year
base["pool_start_month"] = base["pool_start"].dt.month
base["pool_start_dayofweek"] = base["pool_start"].dt.dayofweek
base["pool_start_hour"] = base["pool_start"].dt.hour
base["is_post_event"] = (base["pool_start"] >= EVENT_DATE).astype(int)


def gini(values):
    values = np.asarray([v for v in values if pd.notna(v)], dtype=float)
    if len(values) == 0 or values.sum() <= 0:
        return np.nan
    values = np.sort(values)
    n = len(values)
    return float((2 * np.arange(1, n + 1) @ values) / (n * values.sum()) - (n + 1) / n)


def count_prior_within(group, window):
    times = group["pool_start"].to_numpy()
    counts = []
    left = 0
    for right, t in enumerate(times):
        while left < right and (t - times[left]) > window:
            left += 1
        counts.append(right - left)
    return pd.Series(counts, index=group.index)


if has_event_level:
    wallet_col = "user" if early_events["user"].notna().any() else "wallet"
    amount_col = "volume_usd" if early_events["volume_usd"].notna().any() else "amount"
    early_events["_tx_key"] = early_events["tx_id"].fillna(early_events["signature"]).fillna(
        early_events["pool_id"] + "::" + early_events.index.astype(str)
    )
    event_type = early_events["event_type"].astype(str).str.lower()
    early_events["_is_add"] = event_type.str.contains("add|deposit|mint|buy", regex=True, na=False)
    early_events["_is_remove"] = event_type.str.contains("remove|withdraw|burn|sell", regex=True, na=False)

    wallet_counts = (
        early_events.dropna(subset=[wallet_col])
        .groupby(["pool_id", wallet_col])
        .size()
        .rename("wallet_tx_count")
        .reset_index()
    )
    wallet_summary = (
        wallet_counts.groupby("pool_id")["wallet_tx_count"]
        .agg(
            top_wallet_share=lambda s: float(s.max() / s.sum()) if s.sum() > 0 else np.nan,
            wallet_gini=gini,
        )
        .reset_index()
    )

    early_num = (
        early_events.groupby("pool_id")
        .agg(
            tx_count=("_tx_key", "nunique"),
            unique_users=(wallet_col, "nunique"),
            early_volume=(amount_col, "sum"),
            add_tx_count=("_is_add", "sum"),
            remove_tx_count=("_is_remove", "sum"),
        )
        .reset_index()
    )
    early_num = early_num.merge(wallet_summary, on="pool_id", how="left")
    
    interarrival = []
    for pool, sub in early_events.sort_values("timestamp").groupby("pool_id"):
        diffs = sub["timestamp"].diff().dt.total_seconds().dropna()
        interarrival.append(
            {
                "pool_id": pool,
                "interarrival_mean_sec": diffs.mean() if len(diffs) else np.nan,
                "interarrival_std_sec": diffs.std() if len(diffs) > 1 else 0.0,
            }
        )
    interarrival = pd.DataFrame(interarrival)
else:
    early_num = pd.DataFrame({"pool_id": base["pool_id"]})
    for col in [
        "tx_count",
        "unique_users",
        "early_volume",
        "add_tx_count",
        "remove_tx_count",
        "top_wallet_share",
        "wallet_gini",
        "interarrival_mean_sec",
        "interarrival_std_sec",
    ]:
        early_num[col] = np.nan
    interarrival = early_num[["pool_id", "interarrival_mean_sec", "interarrival_std_sec"]]

early_features = base.merge(
    early_num.drop(columns=["interarrival_mean_sec", "interarrival_std_sec"], errors="ignore"),
    on="pool_id",
    how="left",
)
early_features = early_features.merge(interarrival, on="pool_id", how="left")

early_features["tx_rate_per_min"] = early_features["tx_count"] / EARLY_WINDOW_MINUTES
early_features["volume_rate_per_min"] = early_features["early_volume"] / EARLY_WINDOW_MINUTES
early_features["early_add_remove_ratio"] = early_features["add_tx_count"] / early_features["remove_tx_count"].clip(lower=1)

# Historical creator and burst features. These are backward-looking only.
early_features = early_features.sort_values("pool_start").reset_index(drop=True)
early_features["creator_total_pools"] = early_features.groupby("creator").cumcount().fillna(0)
early_features["prior_token_seen_count"] = early_features.groupby("token_mint").cumcount()
early_features["prior_creator_seen_count"] = early_features["creator_total_pools"]
early_features["creator_recent_activity"] = (
    early_features.groupby("creator", group_keys=False)
    .apply(lambda g: count_prior_within(g, pd.Timedelta(hours=24)))
    .sort_index()
)
early_features["pools_created_same_creator_10min"] = (
    early_features.groupby("creator", group_keys=False)
    .apply(lambda g: count_prior_within(g, pd.Timedelta(minutes=10)))
    .sort_index()
)
early_features["pools_created_global_5min"] = count_prior_within(early_features, pd.Timedelta(minutes=5))

print("Event-level feature mode:", has_event_level)
print(f"Early feature rows: {len(early_features):,}")
display(early_features.head())


Event-level feature mode: False
Early feature rows: 75,520


,pool_id,token_mint,creator,pool_start,pool_start_year,pool_start_month,pool_start_dayofweek,pool_start_hour,is_post_event,tx_count,unique_users,early_volume,interarrival_mean_sec,interarrival_std_sec,tx_rate_per_min,volume_rate_per_min,prior_token_seen_count,prior_creator_seen_count
0,4vWJYxLx9F7WPQeeYzg9cxhDeaPjwruZXCffaSknWFxy,2FPyTwcZLUg1MDrwsyoP4D6s1tM7hAkHYRjkNb5w6Pxk,None,2021-02-14 21:09:21,2021,2,6,21,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0
1,DY8qBwVGLeLJSrWib7L16mL7oB4HNAQ2f9yiYWKof54v,2FPyTwcZLUg1MDrwsyoP4D6s1tM7hAkHYRjkNb5w6Pxk,None,2021-02-14 21:09:45,2021,2,6,21,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0.0
2,GTy6wKTohJUNyL2DPozFqQnmW132oAemPQejXmCQusSR,BQcdHdAQW1hczDbBi9hiegXAR7A98Q9jx3X3iBBBDiq4,None,2021-02-14 21:10:44,2021,2,6,21,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0
3,Fz6yRGsNiXK7hVu4D2zvbwNXW8FQvyJ5edacs3piR1P7,2FPyTwcZLUg1MDrwsyoP4D6s1tM7hAkHYRjkNb5w6Pxk,None,2021-02-14 21:10:48,2021,2,6,21,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0.0
4,6fTRDD7sYxCN7oyoSQaN1AWC3P2m8A6gVZzGrpej9DvL,EPjFWdd5AufqSSqeM2qN1xzybapC8G4wEGGkZwyTDt1v,None,2021-02-14 21:34:21,2021,2,6,21,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0


## Step 4 - Graph Features From Early Information Only

Nodes are pools. Edges connect pools that share token mint or the same early start-time window.


In [29]:
def add_group_edges(G, pools, weight, max_group_size=250):
    pools = list(dict.fromkeys([p for p in pools if pd.notna(p)]))
    if len(pools) < 2:
        return
    if len(pools) > max_group_size:
        pools = pools[:max_group_size]
    for i, u in enumerate(pools):
        for v in pools[i + 1:]:
            if G.has_edge(u, v):
                G[u][v]["weight"] += weight
            else:
                G.add_edge(u, v, weight=weight)


def build_early_graph(features: pd.DataFrame) -> nx.Graph:
    g = nx.Graph()
    g.add_nodes_from(features["pool_id"])

    for _, pools in features.groupby("token_mint")["pool_id"]:
        add_group_edges(g, pools, weight=2)

    tmp = features.copy()
    tmp["early_time_window"] = tmp["pool_start"].dt.floor(f"{EARLY_WINDOW_MINUTES}min")
    for _, pools in tmp.groupby("early_time_window")["pool_id"]:
        add_group_edges(g, pools, weight=1)

    return g


def graph_features(features: pd.DataFrame, g: nx.Graph) -> pd.DataFrame:
    degree = dict(g.degree())
    clustering = nx.clustering(g)
    component_size = {}
    for comp in nx.connected_components(g):
        size = len(comp)
        for node in comp:
            component_size[node] = size

    return pd.DataFrame(
        {
            "pool_id": features["pool_id"],
            "graph_degree": features["pool_id"].map(degree).fillna(0).astype(float),
            "graph_clustering": features["pool_id"].map(clustering).fillna(0).astype(float),
            "graph_component_size": features["pool_id"].map(component_size).fillna(1).astype(float),
        }
    )


early_graph = build_early_graph(early_features)
graph_df = graph_features(early_features, early_graph)
early_features = early_features.merge(graph_df, on="pool_id", how="left")

print(f"Graph nodes: {early_graph.number_of_nodes():,}")
print(f"Graph edges: {early_graph.number_of_edges():,}")
display(graph_df.head())


Graph nodes: 75,520
Graph edges: 951,487


,pool_id,graph_degree,graph_clustering,graph_component_size
0,4vWJYxLx9F7WPQeeYzg9cxhDeaPjwruZXCffaSknWFxy,10.0,0.844444,52468.0
1,DY8qBwVGLeLJSrWib7L16mL7oB4HNAQ2f9yiYWKof54v,10.0,0.844444,52468.0
2,GTy6wKTohJUNyL2DPozFqQnmW132oAemPQejXmCQusSR,5.0,0.400000,52468.0
3,Fz6yRGsNiXK7hVu4D2zvbwNXW8FQvyJ5edacs3piR1P7,10.0,0.844444,52468.0
4,6fTRDD7sYxCN7oyoSQaN1AWC3P2m8A6gVZzGrpej9DvL,250.0,0.992000,52468.0


## Steps 5-6 - Remove Leakage And Merge Labels

The leak guard rejects label-only, full-lifecycle, future-derived, text identifier, and timestamp columns before modeling.


In [30]:
dataset = early_features.merge(labels, on="pool_id", how="left")

LEAKAGE_PATTERNS = (
    "ADD_TO_REMOVE_RATIO",
    "add_to_remove_ratio",
    "lifespan",
    "INACTIVITY_STATUS",
    "inactivity_status",
    "last_",
    "TOTAL_",
    "NUM_LIQUIDITY",
    "removed_liquidity",
    "added_liquidity",
)

def is_leaky_column(col):
    return col in LABEL_ONLY_COLUMNS or any(pattern in col for pattern in LEAKAGE_PATTERNS)


non_feature_cols = {
    "pool_id",
    "token_mint",
    "creator",
    "pool_start",
    "is_rug",
    "has_label",
}

feature_cols = [
    col
    for col in dataset.columns
    if col not in non_feature_cols
    and not is_leaky_column(col)
    and pd.api.types.is_numeric_dtype(dataset[col])
    and dataset[col].notna().any()
]

leaked = [col for col in feature_cols if is_leaky_column(col)]
if leaked:
    raise ValueError(f"Leakage columns escaped into features: {leaked}")

dataset.to_parquet(OUT_DIR / "early_features.parquet", index=False)

print(f"Feature columns ({len(feature_cols)}):")
print(feature_cols)
print(f"Saved: {OUT_DIR / 'early_features.parquet'}")


Feature columns (10):
['pool_start_year', 'pool_start_month', 'pool_start_dayofweek', 'pool_start_hour', 'is_post_event', 'prior_token_seen_count', 'prior_creator_seen_count', 'graph_degree', 'graph_clustering', 'graph_component_size']
Saved: data\early_prediction\early_features.parquet


## Step 7 - Time-Based Splits

Validation:
- train: 2021-01-01 through 2023-12-31
- validate: 2024-01-01 through 2024-12-31

Final model:
- train: 2021-01-01 through 2024-12-31
- forward test/prediction: 2025 only


In [31]:
labeled = dataset[dataset["has_label"]].copy()
labeled["year"] = labeled["pool_start"].dt.year

train_2021_2023 = labeled[(labeled["pool_start"] >= "2021-01-01") & (labeled["pool_start"] < "2024-01-01")].copy()
valid_2024 = labeled[(labeled["pool_start"] >= "2024-01-01") & (labeled["pool_start"] < "2025-01-01")].copy()
train_2021_2024 = labeled[(labeled["pool_start"] >= "2021-01-01") & (labeled["pool_start"] < "2025-01-01")].copy()
forward_2025 = dataset[(dataset["pool_start"] >= "2025-01-01") & (dataset["pool_start"] < "2026-01-01")].copy()

split_summary = pd.DataFrame(
    [
        {"split": "train_2021_2023", "rows": len(train_2021_2023), "labeled": int(train_2021_2023["has_label"].sum()), "rug_rate": train_2021_2023["is_rug"].mean()},
        {"split": "validate_2024", "rows": len(valid_2024), "labeled": int(valid_2024["has_label"].sum()), "rug_rate": valid_2024["is_rug"].mean()},
        {"split": "final_train_2021_2024", "rows": len(train_2021_2024), "labeled": int(train_2021_2024["has_label"].sum()), "rug_rate": train_2021_2024["is_rug"].mean()},
        {"split": "forward_2025", "rows": len(forward_2025), "labeled": int(forward_2025["has_label"].sum()), "rug_rate": forward_2025.loc[forward_2025["has_label"], "is_rug"].mean() if forward_2025["has_label"].any() else np.nan},
    ]
)
display(split_summary)
split_summary.to_csv(OUT_DIR / "split_summary.csv", index=False)


,split,rows,labeled,rug_rate
0,train_2021_2023,9563,9563,0.003765
1,validate_2024,53957,53957,0.019516
2,final_train_2021_2024,63520,63520,0.017144
3,forward_2025,11235,0,NaN


## Steps 8-9 - Modeling And Evaluation

Baseline model: RandomForest. Optional gradient-boosting libraries are intentionally not required.


Threshold tuning starts at `0.02` because the positive class is rare. Metrics include ROC-AUC, PR-AUC, precision, recall, and F1 for each tested threshold.


In [32]:
def can_train(train_df, name):
    if len(train_df) == 0:
        print(f"{name}: no rows")
        return False
    if train_df["is_rug"].nunique() < 2:
        print(f"{name}: needs both classes")
        return False
    return True


def make_model():
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=400,
                    min_samples_leaf=2,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )


def threshold_sweep(model, df, split_name, thresholds=None):
    thresholds = thresholds if thresholds is not None else [0.005, 0.01, 0.02, 0.05, 0.10, 0.20]
    rows = []
    for threshold in thresholds:
        row, _ = evaluate(model, df, split_name, threshold=threshold)
        rows.append(row)
    return pd.DataFrame(rows)


def evaluate(model, df, split_name, threshold=PREDICTION_THRESHOLD):
    y_true = df["is_rug"].astype(int)
    y_score = model.predict_proba(df[feature_cols])[:, 1]
    y_pred = (y_score >= threshold).astype(int)

    metrics = {
        "split": split_name,
        "rows": int(len(df)),
        "rug_rows": int(y_true.sum()),
        "positive_rate": float(y_true.mean()) if len(df) else None,
        "roc_auc": float(roc_auc_score(y_true, y_score)) if y_true.nunique() == 2 else None,
        "pr_auc": float(average_precision_score(y_true, y_score)) if y_true.nunique() == 2 else None,
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "threshold": float(threshold),
    }
    pred = df[["pool_id", "token_mint", "pool_start", "is_rug", "has_label"]].copy()
    pred["score"] = y_score
    pred["prediction"] = y_pred
    pred["split"] = split_name
    return metrics, pred


metrics = {
    "config": {
        "early_window_minutes": EARLY_WINDOW_MINUTES,
        "event_date": str(EVENT_DATE.date()),
        "model": "RandomForestClassifier",
        "leakage_policy": "label-only/full-lifecycle columns dropped before modeling",
    },
    "validation_2024": None,
    "forward_2025": None,
    "notes": [],
}

if not feature_cols:
    raise ValueError("No leakage-safe numeric features are available.")

validation_predictions = pd.DataFrame()
final_model = None

if can_train(train_2021_2023, "Validation model") and len(valid_2024) > 0:
    validation_model = make_model()
    validation_model.fit(train_2021_2023[feature_cols], train_2021_2023["is_rug"].astype(int))
    val_metrics, validation_predictions = evaluate(validation_model, valid_2024, "validate_2024", threshold=PREDICTION_THRESHOLD)
    validation_thresholds = threshold_sweep(validation_model, valid_2024, "validate_2024")
    metrics["validation_threshold_sweep"] = validation_thresholds.to_dict(orient="records")
    metrics["validation_2024"] = val_metrics
    display(pd.DataFrame([val_metrics]))
else:
    metrics["notes"].append("Validation skipped: train_2021_2023 or validate_2024 lacks rows/classes.")

if can_train(train_2021_2024, "Final model"):
    final_model = make_model()
    final_model.fit(train_2021_2024[feature_cols], train_2021_2024["is_rug"].astype(int))
    with open(OUT_DIR / "model.pkl", "wb") as f:
        pickle.dump({"model": final_model, "feature_cols": feature_cols, "config": metrics["config"]}, f)
    print(f"Saved: {OUT_DIR / 'model.pkl'}")
else:
    metrics["notes"].append("Final model skipped: train_2021_2024 lacks rows/classes.")


,split,rows,rug_rows,positive_rate,roc_auc,pr_auc,precision,recall,f1
0,validate_2024,53957,1053,0.019516,0.533352,0.021953,0.0,0.0,0.0


Saved: data\early_prediction\model.pkl


## 2025 Forward Prediction

If 2025 labels exist, compute true forward metrics. If they do not exist, still produce scored predictions and record that 2025 evaluation is unavailable.


In [33]:
predictions_2025 = pd.DataFrame()
if final_model is not None and len(forward_2025) > 0:
    y_score = final_model.predict_proba(forward_2025[feature_cols])[:, 1]
    y_pred = (y_score >= PREDICTION_THRESHOLD).astype(int)
    predictions_2025 = forward_2025[["pool_id", "token_mint", "creator", "pool_start", "is_rug", "has_label"]].copy()
    predictions_2025["score"] = y_score
    predictions_2025["prediction"] = y_pred
    predictions_2025["threshold"] = PREDICTION_THRESHOLD
    predictions_2025.to_parquet(OUT_DIR / "predictions_2025.parquet", index=False)

    labeled_2025 = predictions_2025[predictions_2025["has_label"]].copy()
    if len(labeled_2025) > 0 and labeled_2025["is_rug"].nunique() == 2:
        y_true = labeled_2025["is_rug"].astype(int)
        metrics["forward_2025"] = {
            "split": "forward_2025",
            "rows": int(len(labeled_2025)),
            "rug_rows": int(y_true.sum()),
            "positive_rate": float(y_true.mean()),
            "roc_auc": float(roc_auc_score(y_true, labeled_2025["score"])),
            "pr_auc": float(average_precision_score(y_true, labeled_2025["score"])),
            "precision": float(precision_score(y_true, labeled_2025["prediction"], zero_division=0)),
            "recall": float(recall_score(y_true, labeled_2025["prediction"], zero_division=0)),
            "f1": float(f1_score(y_true, labeled_2025["prediction"], zero_division=0)),
        }
    else:
        metrics["notes"].append(
            "2025 predictions saved, but true forward evaluation is unavailable because 2025 files do not contain complete label-only fields."
        )

    print(f"Saved: {OUT_DIR / 'predictions_2025.parquet'}")
    display(predictions_2025.head())
else:
    metrics["notes"].append("2025 prediction skipped: no final model or no 2025 rows.")


Saved: data\early_prediction\predictions_2025.parquet


,pool_id,token_mint,creator,pool_start,is_rug,has_label,score,prediction
63520,EPjFWdd5AufqSSqeM2qN1xzybapC8G4wEGGkZwyTDt1v::...,EPjFWdd5AufqSSqeM2qN1xzybapC8G4wEGGkZwyTDt1v,whirlpool,2025-01-01,0,False,0.006999,0
63521,Es9vMFrzaCERmJfrF4H2FYD4KCoNkY11McCe8BenwNYB::...,Es9vMFrzaCERmJfrF4H2FYD4KCoNkY11McCe8BenwNYB,whirlpool,2025-01-01,0,False,0.004586,0
63522,25hAyBQfoDhfWx9ay6rarbgvWGwDdNqcHsXS3jQ3mTDJ::...,25hAyBQfoDhfWx9ay6rarbgvWGwDdNqcHsXS3jQ3mTDJ,raydium,2025-01-01,0,False,0.006459,0
63523,JUPyiwrYJFskUPiHa7hkeR8VUtAeFoSYbKedZNsDvCN::w...,JUPyiwrYJFskUPiHa7hkeR8VUtAeFoSYbKedZNsDvCN,whirlpool,2025-01-01,0,False,0.067771,0
63524,GFUgXbMeDnLkhZaJS3nYFqunqkFNMRo9ukhyajeXpump::...,GFUgXbMeDnLkhZaJS3nYFqunqkFNMRo9ukhyajeXpump,raydium,2025-01-01,0,False,0.178337,0


## Step 10 - Outputs

Required outputs:

- `model.pkl`
- `early_features.parquet`
- `predictions_2025.parquet`
- `metrics.json`


In [34]:
with open(OUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))
print("\nOutput files:")
for path in sorted(OUT_DIR.glob("*")):
    print(" -", path)


{
  "config": {
    "early_window_minutes": 30,
    "event_date": "2024-07-01",
    "model": "RandomForestClassifier",
    "leakage_policy": "label-only/full-lifecycle columns dropped before modeling"
  },
  "validation_2024": {
    "split": "validate_2024",
    "rows": 53957,
    "rug_rows": 1053,
    "positive_rate": 0.01951554015234353,
    "roc_auc": 0.533352309093904,
    "pr_auc": 0.021953389905134662,
    "precision": 0.0,
    "recall": 0.0,
    "f1": 0.0
  },
  "forward_2025": null,
  "notes": [
    "2025 predictions saved, but true forward evaluation is unavailable because 2025 files do not contain complete label-only fields."
  ]
}

Output files:
 - data\early_prediction\early_features.parquet
 - data\early_prediction\metrics.json
 - data\early_prediction\model.pkl
 - data\early_prediction\predictions_2025.parquet
 - data\early_prediction\split_summary.csv
